## Growth Predictor Model Training

Forecasts the growth rate of an asset by 1 year, and is designed to provide a float value which fits directly into the Intrinsic Value Analyzer model's input data.

Collected data:
- Tickers, and end-of-year share prices from 2019 till 2023
- model trained on 4 CAGR figures, to forecast the 5th CAGR, next years
- model test data (labels) will come from 2024 end-of-year price
- label calculated from 2023 - 2024 price change or CAGR

#### Imports

In [77]:
from os import path
from csv import DictReader

import logging
import pandas as pd
import numpy as np
import yfinance as yf

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error
from xgboost import XGBRegressor

#### Loading Data from API

In [21]:
data_file = path.join("..", "data")
snp500_path = path.join(data_file, "constituents.csv")
nasdaq_path = path.join(data_file, "nasdaq-listed.csv")
output_path = path.join(data_file, "tickers.csv")
collected_tks = []
existing_tks = set()


with open(snp500_path, mode="r", encoding="utf-8") as f:
    reader = DictReader(f)
    for row in reader:
        ticker = row["Symbol"].strip().upper()
        if ticker and ticker not in existing_tks:
            collected_tks.append(ticker)
            existing_tks.add(ticker)


nasdaq_added_count = 0
with open(nasdaq_path, mode="r", encoding="utf-8") as f:
    reader = DictReader(f)
    for row in reader:
        if nasdaq_added_count >= 500: 
            break
        ticker = row["Symbol"].strip().upper()
        if ticker and ticker not in existing_tks:
            collected_tks.append(ticker)
            existing_tks.add(ticker)
            nasdaq_added_count += 1


print(f"Tickers loaded from {snp500_path} and {nasdaq_path}.")

Tickers loaded from ..\data\constituents.csv and ..\data\nasdaq-listed.csv.


In [22]:
logger = logging.getLogger('yfinance')
logger.setLevel(logging.CRITICAL)
all_price_records = []


try:
    data = yf.download(collected_tks, start="2019-01-09", 
                       end="2025-01-08", group_by='column', progress=False)
    closing = data['Close'] if isinstance(data.columns, pd.MultiIndex) else data
    closing.index = pd.to_datetime(closing.index)
    end_prices = closing.groupby(closing.index.year).last()
    final_df = end_prices.T
    final_df.reset_index(inplace=True)
    final_df.rename(columns={'index': 'Ticker'}, inplace=True)
    final_df.columns = [str(col) for col in final_df.columns]
    columns_order = ['Ticker', '2019', '2020', '2021', '2022', '2023', '2024']
    final_df = final_df.reindex(columns=columns_order)
    final_df.dropna(subset=columns_order, inplace=True)
    final_df.to_csv(output_path, index=False)
    print(f"Saved {len(final_df)} processed tickers to {output_path}")
    print(f"Also added {nasdaq_added_count} tickers from NASDAQ.")
except Exception as e:
    print(f"Error downloading data for multiple tickers: {e}")

Saved 680 processed tickers to ..\data\tickers.csv
Also added 500 tickers from NASDAQ.


#### Calculating CAGR and Scaling ML Data

In [85]:
data_file = path.join("..", "data")
df = pd.read_csv(path.join(data_file, "tickers.csv"))
year_columns = ['2019', '2020', '2021', '2022', '2023', '2024']
cagr_matrix = df[year_columns].pct_change(axis=1).iloc[:, 1:]

cagr_map = {}
for index, row in df.iterrows():
    ticker = row['Ticker']
    cagr_map[ticker] = cagr_matrix.iloc[index].tolist()
    
print("Calculated CAGR Arrays:") # pre-scaling checks
for ticker, cagr_array in list(cagr_map.items())[:3]:
    formatted_array = [round(val, 4) for val in cagr_array]
    print(f"'{ticker}': {formatted_array}")

Calculated CAGR Arrays:
'A': [0.3979, 0.3551, -0.0552, -0.0642, -0.027]
'AACG': [-0.125, -0.1008, 0.1869, -0.0866, -0.2672]
'AADR': [0.1313, 0.0648, -0.2292, 0.1867, 0.2459]


In [86]:
X_list = []
y_list = []
tickers_list = []

for ticker, cagr_array in cagr_map.items():
    features_4yr = cagr_array[:4]
    mean_cagr = np.mean(features_4yr)
    std_cagr  = np.std(features_4yr)
    extended_features = features_4yr + [mean_cagr, std_cagr]
    X_list.append(extended_features) # features: 4 CAGRs (2019 --> 2023)
    y_list.append(cagr_array[4])     # label: 5th CAGR (2023 --> 2024)
    tickers_list.append(ticker)

X = np.array(X_list)
y = np.array(y_list)
tk_array = np.array(tickers_list)
X_train, X_test, y_train, y_test, tk_train, tk_test = train_test_split(
    X, y, tk_array, test_size=0.35, random_state=42
)


#### Model Training

In [87]:
# XGBoost is efficient and dominates short-sequence time-series forecasting
# This model is lightweight and can be trained quickly on a standard CPU
xgr_model = XGBRegressor(
    n_estimators=250, learning_rate=0.03,
    max_depth=3, subsample=0.7,
    colsample_bytree=0.6, random_state=42
)

xgr_model.fit(X_train, y_train)
predicts = xgr_model.predict(X_test)

#### Model Evaluation, With Tolerances

In [88]:
rmse = np.sqrt(mean_squared_error(y_test, predicts))
mae = mean_absolute_error(y_test, predicts)
tolerance = 0.25
within_tolerance = np.abs(predicts - y_test) <= tolerance
tolerance_accuracy = np.mean(within_tolerance) * 100

print("--- Model Evaluation ---")
print(f"Root Mean Squared Error (RMSE): {rmse:.4f}")
print(f"Mean Absolute Error (MAE): {mae:.4f}")
print(f"Accuracy within {tolerance*100}% Tolerance: {tolerance_accuracy:.2f}%")
print("\nPredictions (Actual vs Predicted):")
for i in range(150, min(155, len(y_test))):
    print(f"Ticker: {tk_test[i]} | Actual: {y_test[i]:.4f} | Predicted: {predicts[i]:.4f}")

--- Model Evaluation ---
Root Mean Squared Error (RMSE): 0.4946
Mean Absolute Error (MAE): 0.3129
Accuracy within 25.0% Tolerance: 59.66%

Predictions (Actual vs Predicted):
Ticker: OKE | Actual: 0.5010 | Predicted: 0.2393
Ticker: HUM | Actual: -0.4396 | Predicted: 0.0933
Ticker: MRK | Actual: -0.0626 | Predicted: 0.1441
Ticker: EQT | Actual: 0.2141 | Predicted: 0.2439
Ticker: PG | Actual: 0.1725 | Predicted: 0.1225
